# Procesado de datos
En este notebook se busca obtener un dataset adecuado para transformarlo y entrenar un modelo de inteligencia artificial. 
Importamos librerias de manipulación de datos `pandas` y funciones matemáticas `numpy`. 

In [20]:
import pandas as pd
import numpy as np

In [21]:
df_expenses = pd.read_csv("../data/raw/Expenses_clean.csv")

## Revisión del dataset  

Revisamos el dataset:  
- Columnas y primeros datos  
- Dimensiones, tipo de dato y si hay valores nulos
- Ver los tipos valores únicos transacciones, ya que esto se va a tratar para categorizar 


In [22]:
df_expenses.head(5)

,date_time,category,account,amount,currency,tags
0,2025-11-30 00:00:00,Health,acct_1,114.0,BYN,tag_1
1,2025-11-29 00:00:00,Food,acct_1,5.0,BYN,tag_1
2,2025-11-27 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
3,2025-11-27 00:00:00,Cafe,acct_1,10.0,BYN,tag_1
4,2025-11-27 00:00:00,Public transport,acct_2,1.0,BYN,tag_1


In [23]:
df_expenses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 938 entries, 0 to 937
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date_time  938 non-null    object 
 1   category   938 non-null    object 
 2   account    938 non-null    object 
 3   amount     938 non-null    float64
 4   currency   938 non-null    object 
 5   tags       938 non-null    object 
dtypes: float64(1), object(5)
memory usage: 44.1+ KB


## Revisamos las categorías únicas de las transacciones en el dataframe.  

In [24]:
df_expenses['category'].unique()

array(['Health', 'Food', 'Public transport', 'Cafe', 'Taxi', 'Gifts',
       'Loan given', 'Leisure', 'Clothes', 'Bought for myself', 'Other',
       'University', 'Job', 'Fines'], dtype=object)

## Exploración de las columnas restantes

Las columnas `df_expenses[['account', 'tags']]` son datos anonimizados, no contienen información valiosa

In [25]:
df_expenses[['account', 'tags', 'category']].head(5)

,account,tags,category
0,acct_1,tag_1,Health
1,acct_1,tag_1,Food
2,acct_2,tag_1,Public transport
3,acct_1,tag_1,Cafe
4,acct_2,tag_1,Public transport


In [26]:
df_expenses['account'].unique()

array(['acct_1', 'acct_2', 'acct_3'], dtype=object)

In [27]:
df_expenses['tags'].unique()

array(['tag_1', 'tag_2', 'tag_3', 'tag_4', '2th_work', 'tag_5', 'tag_6'],
      dtype=object)

## Seleccionar datos
Hacemos un dataframe con los datos que necesitamos

In [28]:
df_expenses = pd.DataFrame(df_expenses['category'])
df_expenses.head()

,category
0,Health
1,Food
2,Public transport
3,Cafe
4,Public transport


## Quitamos espacios al inicio y final en cada elemento de la columna `df_expenses['category']`

In [29]:
df_expenses['category'] = df_expenses['category'].str.strip()

## Creamos un diccionario para mapear los las categorias existentes  a las categorias que necesitamos para el proyecto 

In [30]:
categorias_esp = {
    'Food': 'Alimentacion', 'Cafe': 'Alimentacion',
    'Public transport': 'Transporte', 'Taxi': 'Transporte',
    'Health': 'Salud',
    'University': 'Educacion',
    'Leisure': 'Ocio', 'Clothes': 'Ocio', 'Gifts': 'Ocio', 'Bought for myself': 'Ocio',
    'Fines': 'Servicios', 'Loan given': 'Servicios', 'Job': 'Servicios', 'Other': 'Servicios'}
categorias_agrupadas = {
    'Food': 'Food', 'Cafe': 'Food',
    'Public transport': 'Public transport', 'Taxi': 'Public transport',
    'Health': 'Health',
    'University': 'University',
    'Leisure': 'Leisure', 'Clothes': 'Leisure', 'Gifts': 'Leisure', 'Bought for myself': 'Leisure',
    'Fines': 'Fines', 'Loan given': 'Fines', 'Job': 'Fines', 'Other': 'Fines'}

In [31]:
df_expenses['categorias'] = df_expenses['category'].map(categorias_esp)
df_expenses['categorias_agrup'] = df_expenses['category'].map(categorias_agrupadas)
df_expenses = df_expenses.dropna(subset=['categorias'])

In [32]:
df_expenses['category'].unique()

array(['Health', 'Food', 'Public transport', 'Cafe', 'Taxi', 'Gifts',
       'Loan given', 'Leisure', 'Clothes', 'Bought for myself', 'Other',
       'University', 'Job', 'Fines'], dtype=object)

In [33]:
df_expenses['categorias'].unique()

array(['Salud', 'Alimentacion', 'Transporte', 'Ocio', 'Servicios',
       'Educacion'], dtype=object)

In [34]:
df_expenses['categorias_agrup'].unique()

array(['Health', 'Food', 'Public transport', 'Leisure', 'Fines',
       'University'], dtype=object)

## Inyeccion de la categoria vivienda y categorias en general

In [35]:
descripciones_base = {
    'Food': [
        'Compra en supermercado', 'Cena en restaurante', 'Comida rapida', 
        'Despensa quincenal', 'Panaderia', 'Cafe espresso', 'Starbucks', 
        'Cafeteria local', 'Te y galletas', 'Matcha latte'
    ],
    
    'Public transport': [
        'Recarga tarjeta metro', 'Boleto de autobus', 'Tren suburbano', 
        'Cablebus', 'Taxi', 'Microbus'
    ],
    
    'Health': [
        'Consulta medica', 'Farmacia', 'Examen de laboratorio', 
        'Seguro de gastos medicos', 'Doctor'
    ],
    
    'Housing': [
        'Pago de renta', 'Mantenimiento del edificio', 'Hipoteca', 
        'Articulos de limpieza', 'Muebles para sala', 'Reparacion de plomeria', 
        'Ferreteria', 'Pintura para interiores', 'Decoracion del hogar', 
        'Seguro de vivienda'
    ],
    
    'Education': [
        'Colegiatura', 'Inscripcion escolar', 'Utiles escolares', 
        'Curso en linea', 'Libros de texto', 'Certificacion', 
        'Clases particulares', 'Material didactico', 'Mensualidad universidad', 
        'Taller de idiomas'
    ],
    
    'Leisure': [
        'Camiseta', 'Pantalon', 'Gorra', 'Tour por ciudad', 'Visita guiada', 
        'Senderismo', 'Ciclismo', 'Patinaje', 'Natacion', 'Surf', 'Cartera', 
        'Reloj', 'Lentes de sol', 'Joyeria', 'Perfume', 'Crema', 
        'Set de ducha', 'Libro', 'Concierto', 'Boleto de cine', 
        'Obra de teatro', 'Spa'
    ],
    
    'Services': [
        'Recibo de luz', 'Factura de agua', 'Plan de internet', 
        'Telefonia movil', 'Gas natural', 'Suscripcion de streaming', 
        'Mensualidad del gimnasio', 'Mantenimiento de caldera', 
        'Pago de television por cable', 'Servicio de limpieza'
    ]
}
descripciones_base2 = {
    'Food': [
        'Compra en supermercado', 'Cena en restaurante', 'Comida rapida', 
        'Despensa quincenal', 'Panaderia', 'Cafe espresso', 'Starbucks', 
        'Cafeteria local', 'Te y galletas', 'Matcha latte'
    ],
    
    'Public transport': [
        'Recarga tarjeta metro', 'Boleto de autobus', 'Tren suburbano', 
        'Cablebus', 'Taxi', 'Microbus', 'Mexibus', 'Metrobus', 'Trolebus'],
    
    'Health': [
        'Consulta medica', 'Farmacia', 'Examen de laboratorio', 
        'Seguro de gastos medicos', 'Doctor'],
    
    'Housing': [
        'Renta', 'Mantenimiento del edificio', 'Hipoteca', 
        'Articulos de limpieza', 'Muebles para sala', 'Reparacion de plomeria', 
        'Ferreteria', 'Pintura para interiores', 'Decoracion del hogar', 
        'Seguro de vivienda'],
    
    'University': [
        'Colegiatura', 'Inscripcion escolar', 'Utiles escolares', 
        'Curso en linea', 'Libros de texto', 'Certificacion', 
        'Clases particulares', 'Material didactico', 'Mensualidad universidad', 
        'Taller de idiomas'],
    
    'Leisure': [
        'Camiseta', 'Pantalon', 'Gorra', 'Tour por ciudad', 'Visita guiada', 
        'Senderismo', 'Ciclismo', 'Patinaje', 'Natacion', 'Surf', 'Cartera', 
        'Reloj', 'Lentes de sol', 'Joyeria', 'Perfume', 'Crema', 
        'Set de ducha', 'Libro', 'Concierto', 'Boleto de cine', 
        'Obra de teatro', 'Spa'],
    
    'Fines': [
        'Recibo de luz', 'Factura de agua', 'Plan de internet', 
        'Telefonia movil', 'Gas natural', 'Suscripcion de streaming', 
        'Mensualidad del gimnasio', 'Mantenimiento de caldera', 
        'Television por cable', 'Servicio de limpieza']}

### Funcion generadora de texto aleatorio 

In [36]:
def generar_descripcion_realista(categoria):
    if categoria in descripciones_base:
        return np.random.choice(descripciones_base[categoria])
    return f'Pago por concepto de {categoria.lower()}'
def generar_descripcion_realista2(categoria):
    if categoria in descripciones_base2:
        return np.random.choice(descripciones_base2[categoria])
    return f'Pago por concepto de {categoria.lower()}'

In [37]:
df_expenses['descripcion_limpia'] = df_expenses['category'].apply(generar_descripcion_realista)
df_expenses.tail(5)

,category,categorias,categorias_agrup,descripcion_limpia
933,Public transport,Transporte,Public transport,Microbus
934,Public transport,Transporte,Public transport,Taxi
935,Taxi,Transporte,Public transport,Pago por concepto de taxi
936,Taxi,Transporte,Public transport,Pago por concepto de taxi
937,Food,Alimentacion,Food,Cafe espresso


In [38]:
df_expenses['descripcion_limpia'] = df_expenses['categorias_agrup'].apply(generar_descripcion_realista2)
df_expenses.tail(5)

,category,categorias,categorias_agrup,descripcion_limpia
933,Public transport,Transporte,Public transport,Recarga tarjeta metro
934,Public transport,Transporte,Public transport,Trolebus
935,Taxi,Transporte,Public transport,Cablebus
936,Taxi,Transporte,Public transport,Tren suburbano
937,Food,Alimentacion,Food,Despensa quincenal


In [39]:
df_expenses = df_expenses[['descripcion_limpia', 'categorias']]

In [40]:
df_vivienda = pd.DataFrame({'descripcion_limpia': descripciones_base2['Housing'], 'categorias': 'Vivienda'})
df_vivienda

,descripcion_limpia,categorias
0,Renta,Vivienda
1,Mantenimiento del edificio,Vivienda
2,Hipoteca,Vivienda
3,Articulos de limpieza,Vivienda
4,Muebles para sala,Vivienda
5,Reparacion de plomeria,Vivienda
6,Ferreteria,Vivienda
7,Pintura para interiores,Vivienda
8,Decoracion del hogar,Vivienda
9,Seguro de vivienda,Vivienda


In [41]:
df_expenses = pd.concat([df_expenses, df_vivienda], ignore_index=True)
df_expenses

,descripcion_limpia,categorias
0,Doctor,Salud
1,Starbucks,Alimentacion
2,Cablebus,Transporte
3,Cafeteria local,Alimentacion
4,Trolebus,Transporte
...,...,...
943,Reparacion de plomeria,Vivienda
944,Ferreteria,Vivienda
945,Pintura para interiores,Vivienda
946,Decoracion del hogar,Vivienda


In [42]:
df_expenses['categorias'].unique()

array(['Salud', 'Alimentacion', 'Transporte', 'Ocio', 'Servicios',
       'Educacion', 'Vivienda'], dtype=object)

In [43]:
df_expenses['descripcion_limpia'].unique()

array([np.str_('Doctor'), np.str_('Starbucks'), np.str_('Cablebus'),
       np.str_('Cafeteria local'), np.str_('Trolebus'),
       np.str_('Boleto de autobus'), np.str_('Recarga tarjeta metro'),
       np.str_('Microbus'), np.str_('Comida rapida'),
       np.str_('Tren suburbano'), np.str_('Cena en restaurante'),
       np.str_('Metrobus'), np.str_('Mexibus'), np.str_('Te y galletas'),
       np.str_('Taxi'), np.str_('Visita guiada'),
       np.str_('Despensa quincenal'), np.str_('Matcha latte'),
       np.str_('Compra en supermercado'), np.str_('Recibo de luz'),
       np.str_('Consulta medica'), np.str_('Reloj'), np.str_('Panaderia'),
       np.str_('Gorra'), np.str_('Obra de teatro'),
       np.str_('Cafe espresso'), np.str_('Ciclismo'), np.str_('Farmacia'),
       np.str_('Cartera'), np.str_('Spa'), np.str_('Perfume'),
       np.str_('Crema'), np.str_('Patinaje'), np.str_('Telefonia movil'),
       np.str_('Factura de agua'), np.str_('Surf'), np.str_('Libro'),
       np.str_('Gas 

In [46]:
df_expenses.head(5)

,descripcion_limpia,categorias
0,Doctor,Salud
1,Starbucks,Alimentacion
2,Cablebus,Transporte
3,Cafeteria local,Alimentacion
4,Trolebus,Transporte


In [47]:
df_expenses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 948 entries, 0 to 947
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   descripcion_limpia  948 non-null    object
 1   categorias          948 non-null    object
dtypes: object(2)
memory usage: 14.9+ KB


## Guardado del DataFrame 

In [45]:
df_expenses.to_csv('../data/processed/df_gastos.csv', index=False)